# Mô hình dự báo nguy cơ trễ SLA (v4): chạy lại và đánh giá trên Colab / Kaggle

Notebook này **tái lập** pipeline của dự án SME Loan: kiểm tra dữ liệu, huấn luyện lại hai mô hình, đối chiếu với kết quả trong kho mã, rồi chạy thực nghiệm bổ sung (so sánh thuật toán, nhiều hạt giống, khoảng tin cậy, bóc tách, SLA nhận biết vs cố định, độ trễ). Dữ liệu là **mô phỏng** (20.000 hồ sơ), không phải dữ liệu ngân hàng thật.

Chạy từng ô theo thứ tự. Không cần GPU.

## 1. Môi trường (ghi lại làm bằng chứng)

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Hệ điều hành:", platform.platform())
# Mô hình trong dự án được huấn luyện với đúng các phiên bản này:
!pip -q install scikit-learn==1.9.1 pandas==3.1.1 joblib numpy
import sklearn, pandas, numpy
print("scikit-learn:", sklearn.__version__, "| pandas:", pandas.__version__, "| numpy:", numpy.__version__)
# Nếu sklearn không phải 1.9.1: Runtime > Restart session rồi chạy lại ô này.

## 2. Nạp mã và dữ liệu
**Colab:** chọn tệp `ml-experiments-v4.zip` khi hộp thoại hiện ra.
**Kaggle:** thêm tệp zip làm Dataset (Add Data > Upload), rồi bỏ comment đoạn Kaggle bên dưới và comment đoạn Colab.

In [ ]:
# --- Colab ---
from google.colab import files
up = files.upload()
!unzip -q -o ml-experiments-v4.zip -d /content
%cd /content/v3

# --- Kaggle (bỏ comment, và comment đoạn Colab ở trên) ---
# !mkdir -p /kaggle/working/v3 && cd /kaggle/working && unzip -q -o /kaggle/input/<tên-dataset>/ml-experiments-v4.zip -d /kaggle/working
# %cd /kaggle/working/v3

!ls -la . data

## 3. Kiểm tra dữ liệu có giống bản gốc không
Băm SHA-256 phải trùng với giá trị dưới đây (ghi trong tài liệu dự án):
- cases_v4.csv: 833860F5EA7B8B36...
- tham_dinh_timeline_v4.csv: 6DE288F18E4853A7...
- tiep_nhan_timeline_v4.csv: F5EA70B5FCE0774A...

In [ ]:
import hashlib, glob
for f in sorted(glob.glob("data/*.csv")):
    print(f.split("/")[-1].ljust(30), hashlib.sha256(open(f, "rb").read()).hexdigest()[:16].upper())

## 4. Kiểm tra chất lượng dữ liệu (31 mục, kỳ vọng 31/31 đạt)

In [ ]:
!python check_quality_v4.py | tail -8

## 5. Huấn luyện lại hai mô hình và đối chiếu với kết quả trong kho mã
`train_v4.py` ghi đè `results/metrics_v4.json`; ô này lưu bản gốc trước để so sánh.

In [ ]:
import shutil, json, math
shutil.copy("results/metrics_v4.json", "results/metrics_v4_repo.json")
!python train_v4.py | tail -6
new = json.load(open("results/metrics_v4.json")); old = json.load(open("results/metrics_v4_repo.json"))
def walk(a, b, path=""):
    if isinstance(a, dict):
        for k in a: yield from walk(a[k], b.get(k) if isinstance(b, dict) else None, path + "/" + str(k))
    elif isinstance(a, list):
        for i, x in enumerate(a): yield from walk(x, b[i] if isinstance(b, list) and i < len(b) else None, path + f"[{i}]")
    elif isinstance(a, (int, float)): yield path, a, b
bad = [(p, a, b) for p, a, b in walk(old, new) if not (isinstance(b, (int, float)) and math.isclose(a, b, abs_tol=1e-4))]
total = sum(1 for _ in walk(old, new))
print(f"Số giá trị đối chiếu: {total}; lệch: {len(bad)}")
print(bad[:5])
for s in ("tiep_nhan", "tham_dinh"):
    o = new[s]["scenarios"]["C_112_40"]["overall"]
    print(s, "AUC mô hình", o["auc_model"], "| AUC cơ sở", o["auc_baseline"])

## 6. Thực nghiệm bổ sung
Mất khoảng 15 đến 40 phút trên CPU của Colab (rừng ngẫu nhiên và bootstrap là phần nặng). Muốn thử nhanh thì thêm `--quick`.

In [ ]:
!python eval_extra_v4.py

## 7. Tóm tắt kết quả (dán vào bài báo và khóa luận)

In [ ]:
import json, pandas as pd
j = json.load(open("results/experiments_v4.json"))
print("Môi trường:", j["env"]); print("Thời gian chạy (phút):", j["runtime_minutes"])
names = {"rule": "Luật ngưỡng (đã dùng/SLA)", "base": "Đường cơ sở (chỉ đồng hồ SLA)", "lr": "Logistic Regression", "rf": "Random Forest", "gb": "Gradient Boosting"}
for s, st in j["stages"].items():
    print("\n=====", s, "- SLA", st["live_sla"], "- thuật toán đang dùng:", st["chosen"])
    rows = []
    for k in ("rule", "base", "lr", "rf", "gb"):
        m = st["seeds"]["models"][k]
        f = lambda x: f'{x["mean"]:.4f} ± {x["std"]:.4f}'
        rows.append({"Mô hình": names[k], "AUC": f(m["auc"]), **({} if k == "rule" else {"Brier": f(m["brier"]), "Log-loss": f(m["logloss"]), "ECE": f(m["ece"]), "Khẩn: độ chính xác": f(m["urgent_precision"]), "Khẩn: độ phủ": f(m["urgent_recall"]), "Khẩn: báo nhầm": f(m["urgent_false_alarm"])})})
    print(pd.DataFrame(rows).to_string(index=False))
    b = st["bootstrap"]; print("Bootstrap 95%: AUC mô hình", b["auc_model_ci"], "| cơ sở", b["auc_baseline_ci"], "| hiệu", b["auc_diff_ci"], "| tỷ lệ lần mô hình hơn cơ sở", b["share_model_better"])
    print("Bóc tách (AUC trung bình ± độ lệch chuẩn):")
    for v, m in st["seeds"]["ablation"].items(): print("  ", v.ljust(16), f'{m["auc"]["mean"]:.4f} ± {m["auc"]["std"]:.4f}   Brier {m["brier"]["mean"]:.4f}   ECE {m["ece"]["mean"]:.4f}')
    print("SLA nhận biết vs cố định:")
    for L2, v in st["seeds"]["fixed_vs_aware"].items():
        fk = [k for k in v if k.startswith("fixed_at_")][0]
        print(f'  đo ở SLA {L2}: nhận biết AUC {v["aware"]["auc"]["mean"]:.4f} | {fk} AUC {v[fk]["auc"]["mean"]:.4f}')
print("\nĐộ trễ suy luận (ms/hồ sơ):", j["latency_model_only"])

## 7b. Chẩn đoán bổ sung (đợt 2)
- Đường ROC và bảng độ nhạy theo ngưỡng cảnh báo
- Đánh giá phần lý do giải thích
- **Trần lý thuyết:** cho mô hình thấy ba biến ẩn của bộ mô phỏng (chạy lại mô phỏng, có kiểm tra khớp 100% với dữ liệu hiện có)
- Độ nhạy theo giả định mô phỏng (đổi tham số rồi xem kết luận còn đúng không)
- Vẽ các hình (ROC, ngưỡng, bóc tách, SLA nhận biết)
Mất khoảng 10 đến 30 phút trên Colab. Muốn thử nhanh thì thêm `--quick`.

In [ ]:
!python eval_diagnostics_v4.py

## 7c. Tóm tắt chẩn đoán và hình

In [ ]:
import json, glob
d = json.load(open("results/diagnostics_v4.json"))
print("Trần lý thuyết, kiểm tra khớp dữ liệu hiện có:", d["oracle"]["check"])
for s in ("tiep_nhan", "tham_dinh"):
    o = d["oracle"][s]
    print("")
    print("==", s, "- số hồ sơ dùng:", o["cases_used"])
    for k in ("lr_observable", "lr_with_hidden", "gb_observable", "gb_with_hidden"):
        print("  ", k.ljust(16), "AUC %.4f  Brier %.4f" % (o[k]["auc"], o[k]["brier"]))
    r = d["reasons"][s]
    print("  Lý do giải thích, tỷ lệ theo nhóm:", r["share_by_group"], "| dòng có ít nhất 1 lý do:", r["rows_with_at_least_one_reason"], "/", r["rows_explained"])
print("")
print("Độ nhạy theo giả định mô phỏng (AUC mô hình đang dùng trừ AUC đường cơ sở):")
for name, v in d["sensitivity"].items():
    print("  ", name.ljust(40), {k: x["gain_chosen_vs_base"] for k, x in v.items() if isinstance(x, dict)})
from IPython.display import Image, display
for f in sorted(glob.glob("results/figures/*.png")):
    print(f)
    display(Image(f, width=520))

## 8. Tải kết quả về máy (làm bằng chứng)

In [ ]:
!zip -q -r ket-qua-v4.zip results models
from google.colab import files
files.download("ket-qua-v4.zip")   # Kaggle: tệp nằm ở thư mục Output, không cần ô này